# Reasoning, Planning, and Bounded Control

> **The story.** ReAct made action traces inspectable, but inspectable does not mean bounded. Production agents need the same controls long used in workflow engines: budgets, terminal states, retry rules, and cycle detection.
>
> **Where you are.** Notebook 00 gave OrderFlow typed tools. PO `#7298` now requests more sensors than inventory holds. The naive controller keeps checking the same fact because it has no definition of progress.
>
> **Notation.** $s_t$ is workflow state; $a_t$ is the action at step $t$; $b_s$, $b_t$, and $b_c$ are step, token, and cost budgets; $f(a_t, o_t)$ is a cycle fingerprint.

## 0 - The Challenge

> **The mission:** every fixture must terminate in at most eight steps, repeated-action loops must be detected, and solvable cases must remain at or above 90% completion.

```mermaid
flowchart LR
    A["Unavailable inventory"] --> B["Naive ReAct loop"]
    B --> C["Same action repeats"]
    C --> D["Plan + budgets + cycle check"]
    D --> E["Explicit terminal state"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "learning" / "agentic-ai" / "shared" / "__init__.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from the ai-portfolio repository or a descendant directory.")


REPO_ROOT = find_repo_root()
TRACK_DIR = REPO_ROOT / "learning" / "agentic-ai"
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from dataclasses import dataclass, field
from typing import Any

from shared import INVENTORY, SupplierServiceDouble, estimate_tokens, request_by_id, stable_hash

shortage_request = request_by_id("PO-7298")
assert INVENTORY[shortage_request["sku"]]["available"] < shortage_request["quantity"]
print("Walking incident:", shortage_request["email"])

## 1 - Failure First: Plausible Reasoning Can Still Loop

![A repeated inventory loop drains budget on the left while a bounded three-step plan, resource limiters, and cycle detector provide terminal exits on the right](../images/ch01-bounded-control-loop.png)

**Predict:** If the observation never changes, how many times will a controller without a terminal rule call inventory: once, until success, or until an external process kills it?

```mermaid
flowchart LR
    S["Need 15 sensors"] --> C["Check inventory"]
    C --> O["Only 9 available"]
    O --> C
    style S fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style O fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

In [ ]:
# -- Failure first: an observation without a terminal rule ----------------
def unbounded_inventory_loop(request, demonstration_cap=12):
    trace = []
    for step in range(1, demonstration_cap + 1):
        observation = {
            "available": INVENTORY[request["sku"]]["available"],
            "requested": request["quantity"],
        }
        trace.append({"step": step, "action": "check_inventory", "observation": observation})
        if observation["available"] >= observation["requested"]:
            return trace, "ready"
        # No else branch: the controller interprets failure as "try again".
    return trace, "externally_stopped"

naive_trace, naive_terminal = unbounded_inventory_loop(shortage_request)
repeated_calls = sum(event["action"] == "check_inventory" for event in naive_trace)
print(f"Naive terminal state: {naive_terminal}; repeated inventory calls: {repeated_calls}")
assert naive_terminal == "externally_stopped" and repeated_calls == 12
print("Failure observed: rationale did not provide progress or termination.")


## 2 - Plan Before Execute, Then Check Progress

A plan is not hidden chain-of-thought. It is an inspectable task list with preconditions and terminal outcomes. For an inventory shortage, the plan must change tools instead of asking the same question again.

```mermaid
flowchart TD
    P["Plan"] --> I["Inspect inventory"]
    I --> Q{ "Enough stock?" }
    Q -->|"Yes"| F["Price and finish"]
    Q -->|"No"| S["Request supplier quote"]
    S --> A["Needs approval or finish"]
    style P fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style I fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style Q fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Build an explicit plan and a structural cycle fingerprint ------------
def make_plan(request):
    return [
        {"task": "check_inventory", "status": "pending"},
        {"task": "get_fresh_quote", "status": "pending"},
        {"task": "route_approval", "status": "pending"},
    ]

def fingerprint(action, arguments, observation):
    return stable_hash({"action": action, "arguments": arguments, "observation": observation})

plan = make_plan(shortage_request)
print(json.dumps(plan, indent=2))
assert [task["task"] for task in plan] == ["check_inventory", "get_fresh_quote", "route_approval"]
print("PASS: the next useful action remains visible after inventory fails.")


## 3 - Budgets, Retry Policy, and Terminal States

Budgets turn an open-ended loop into a bounded computation. A retry is valid only when the error is transient and the next attempt changes something, such as elapsed backoff or provider choice.

$$
C = \sum_{t=1}^{n}(c_{model,t} + c_{tool,t}) \le b_c
$$

The controller adds model and tool cost per step and must stop before the total exceeds the declared cost budget.

```mermaid
flowchart LR
    A["Action"] --> B{ "Progress?" }
    B -->|"Yes"| C["Continue within budgets"]
    B -->|"No"| D{ "Same fingerprint?" }
    D -->|"Yes"| E["Terminal: cycle detected"]
    D -->|"No"| F["Retry if transient"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Implement the bounded controller --------------------------------------
@dataclass
class Budget:
    max_steps: int = 8
    max_tokens: int = 600
    max_cost: float = 8.0

@dataclass
class RunResult:
    terminal: str
    trace: list[dict[str, Any]] = field(default_factory=list)
    tokens: int = 0
    cost: float = 0.0


def run_bounded(request, budget=Budget(), supplier=None):
    supplier = supplier or SupplierServiceDouble()
    result = RunResult(terminal="running")
    seen = set()
    plan = make_plan(request)
    for step, task in enumerate(plan, 1):
        if step > budget.max_steps:
            result.terminal = "step_budget_exhausted"
            break
        action = task["task"]
        arguments = {"sku": request["sku"], "quantity": request["quantity"]}
        if action == "check_inventory":
            observation = {"available": INVENTORY[request["sku"]]["available"]}
        elif action == "get_fresh_quote":
            quotes = supplier.quotes(request["sku"])
            observation = min(
                [quote for quote in quotes if quote["trusted"] and quote["age_hours"] <= 48],
                key=lambda quote: quote["unit_price"],
            )
        else:
            total = result.trace[-1]["observation"]["unit_price"] * request["quantity"]
            observation = {"route": "auto" if total <= 5000 else "manager" if total <= 25000 else "finance", "total": total}
        current_fingerprint = fingerprint(action, arguments, observation)
        if current_fingerprint in seen:
            result.terminal = "cycle_detected"
            break
        seen.add(current_fingerprint)
        step_tokens = estimate_tokens(json.dumps({"action": action, "observation": observation}))
        step_cost = 1.0
        if result.tokens + step_tokens > budget.max_tokens or result.cost + step_cost > budget.max_cost:
            result.terminal = "budget_exhausted"
            break
        result.tokens += step_tokens
        result.cost += step_cost
        result.trace.append({"step": step, "action": action, "observation": observation})
        task["status"] = "complete"
    else:
        result.terminal = "completed"
    return result

bounded = run_bounded(shortage_request)
print(json.dumps({"terminal": bounded.terminal, "steps": len(bounded.trace), "tokens": bounded.tokens, "cost": bounded.cost}, indent=2))
assert bounded.terminal == "completed" and len(bounded.trace) <= 8


In [ ]:
# -- Prove cycle detection independently of the happy path ----------------
def detect_repeated_actions(events):
    seen = set()
    for event in events:
        key = stable_hash({"action": event["action"], "observation": event["observation"]})
        if key in seen:
            return True
        seen.add(key)
    return False

assert detect_repeated_actions(naive_trace)
assert not detect_repeated_actions(bounded.trace)
print(f"Cycle detected in naive trace: {detect_repeated_actions(naive_trace)}")
print(f"Cycle detected in bounded trace: {detect_repeated_actions(bounded.trace)}")
print("PASS: repeated state-action pairs stop the controller instead of consuming the budget silently.")


### Per-Tool Timeouts and Retries Share the Run Budget

One global timeout creates a bad choice. Tune it for fast inventory checks and a legitimate supplier quote is killed early. Tune it for supplier latency and a stuck inventory call can occupy most of the run. The controller needs different limits for different tools, but those limits must still spend one shared run budget.

The policy belongs to the tool registration, not to text improvised by the model. Each registered tool declares a timeout, an attempt cap, and retry backoff. Before any attempt or retry starts, the controller reserves enough remaining budget for that attempt's declared worst case.

This chapter consumes only a simple `retryable` flag and category from each observation. Chapter 00 owns the fuller error taxonomy; the control loop only decides whether another bounded attempt is permitted.

```mermaid
flowchart LR
    G["One global timeout"] --> F["Fast setting:<br/>slow quote killed"]
    G --> S["Slow setting:<br/>stuck call dominates"]
    R["Registered tool policy"] --> D["Deadline check"]
    D --> A["Bounded attempt"]
    A --> T{ "Retryable?" }
    T -->|"Yes, budget remains"| B["Backoff then retry"]
    T -->|"No or no budget"| X["Terminal state"]
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style S fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style A fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style T fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Across a fast inventory call, a stuck inventory call, and a legitimate 650 ms supplier quote, which policy completes the most useful work without letting the stuck call dominate: 150 ms globally, 700 ms globally, or registered per-tool limits?

In [ ]:
# -- Register typed per-tool policies and simulate bounded execution ---------
@dataclass(frozen=True)
class ToolExecutionPolicy:
    timeout_ms: int
    max_attempts: int
    backoff_ms: int

    def __post_init__(self):
        assert self.timeout_ms > 0
        assert self.max_attempts >= 1
        assert self.backoff_ms >= 0


@dataclass(frozen=True)
class ToolRegistration:
    name: str
    execution_policy: ToolExecutionPolicy


@dataclass(frozen=True)
class SimulatedObservation:
    latency_ms: int
    outcome: str
    retryable: bool
    category: str


@dataclass(frozen=True)
class SimulatedCall:
    label: str
    tool_name: str
    observations: tuple[SimulatedObservation, ...]


TOOL_REGISTRY = {
    "check_inventory": ToolRegistration(
        name="check_inventory",
        execution_policy=ToolExecutionPolicy(timeout_ms=150, max_attempts=1, backoff_ms=0),
    ),
    "get_supplier_quote": ToolRegistration(
        name="get_supplier_quote",
        execution_policy=ToolExecutionPolicy(timeout_ms=700, max_attempts=2, backoff_ms=100),
    ),
}


def execute_simulated_call(call, start_ms, run_budget_ms, policy):
    """Consume modeled latency only; no sleeping, clocks, or network calls."""
    clock_ms = start_ms
    events = []
    attempts = 0
    terminal = "running"

    for attempt in range(1, policy.max_attempts + 1):
        backoff_ms = policy.backoff_ms if attempt > 1 else 0
        declared_worst_case_ms = backoff_ms + policy.timeout_ms
        remaining_ms = run_budget_ms - clock_ms
        if declared_worst_case_ms > remaining_ms:
            terminal = "deadline_prevented_retry" if attempt > 1 else "deadline_prevented_start"
            events.append({
                "kind": "blocked",
                "state": terminal,
                "label": call.label,
                "attempt": attempt,
                "start_ms": clock_ms,
                "end_ms": clock_ms,
            })
            break

        if backoff_ms:
            events.append({
                "kind": "backoff",
                "state": "backoff",
                "label": call.label,
                "attempt": attempt,
                "start_ms": clock_ms,
                "end_ms": clock_ms + backoff_ms,
            })
            clock_ms += backoff_ms

        observation = call.observations[min(attempt - 1, len(call.observations) - 1)]
        timed_out = observation.latency_ms > policy.timeout_ms
        duration_ms = policy.timeout_ms if timed_out else observation.latency_ms
        state = "timeout" if timed_out else observation.outcome
        events.append({
            "kind": "attempt",
            "state": state,
            "category": observation.category,
            "retryable": observation.retryable,
            "label": call.label,
            "attempt": attempt,
            "start_ms": clock_ms,
            "end_ms": clock_ms + duration_ms,
        })
        clock_ms += duration_ms
        attempts += 1

        if state == "success":
            terminal = "completed"
            break
        if not observation.retryable:
            terminal = "tool_timeout" if state == "timeout" else "non_retryable_error"
            break
        if attempt == policy.max_attempts:
            terminal = "tool_timeout" if state == "timeout" else "retry_exhausted"

    return {
        "label": call.label,
        "terminal": terminal,
        "success": terminal == "completed",
        "attempts": attempts,
        "start_ms": start_ms,
        "end_ms": clock_ms,
        "events": events,
    }


def run_policy_batch(label, calls, run_budget_ms, global_timeout_ms=None):
    """Compare one global timeout with policies owned by tool registrations."""
    clock_ms = 0
    call_results = []
    events = []
    for call in calls:
        policy = (
            ToolExecutionPolicy(global_timeout_ms, max_attempts=1, backoff_ms=0)
            if global_timeout_ms is not None
            else TOOL_REGISTRY[call.tool_name].execution_policy
        )
        result = execute_simulated_call(call, clock_ms, run_budget_ms, policy)
        clock_ms = result["end_ms"]
        call_results.append(result)
        events.extend(result["events"])

    return {
        "label": label,
        "successes": sum(result["success"] for result in call_results),
        "attempts": sum(result["attempts"] for result in call_results),
        "consumed_ms": clock_ms,
        "unused_ms": run_budget_ms - clock_ms,
        "run_budget_ms": run_budget_ms,
        "terminal": call_results[-1]["terminal"],
        "events": events,
    }


print("Registered execution policies:")
for registration in TOOL_REGISTRY.values():
    print(f"  {registration.name}: {registration.execution_policy}")
assert TOOL_REGISTRY["check_inventory"].execution_policy.timeout_ms < TOOL_REGISTRY["get_supplier_quote"].execution_policy.timeout_ms
print("PASS: timeout and retry limits are typed registration data, not model-authored arguments.")

In [ ]:
# -- Compare global and per-tool policies with deterministic latency ---------
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

benchmark_calls = (
    SimulatedCall(
        label="inventory: fast",
        tool_name="check_inventory",
        observations=(SimulatedObservation(80, "success", False, "ok"),),
    ),
    SimulatedCall(
        label="inventory: stuck",
        tool_name="check_inventory",
        observations=(SimulatedObservation(900, "transient_error", False, "stuck"),),
    ),
    SimulatedCall(
        label="supplier: legitimate slow",
        tool_name="get_supplier_quote",
        observations=(SimulatedObservation(650, "success", False, "ok"),),
    ),
)

comparison_runs = [
    run_policy_batch("global 150 ms", benchmark_calls, run_budget_ms=1800, global_timeout_ms=150),
    run_policy_batch("global 700 ms", benchmark_calls, run_budget_ms=1800, global_timeout_ms=700),
    run_policy_batch("registered per tool", benchmark_calls, run_budget_ms=1800),
]

retry_call = SimulatedCall(
    label="supplier: retry succeeds",
    tool_name="get_supplier_quote",
    observations=(
        SimulatedObservation(300, "transient_error", True, "transient"),
        SimulatedObservation(450, "success", False, "ok"),
    ),
)
timeout_call = SimulatedCall(
    label="supplier: repeated timeout",
    tool_name="get_supplier_quote",
    observations=(SimulatedObservation(1200, "transient_error", True, "transient"),),
)


def run_single_probe(label, call, run_budget_ms):
    policy = TOOL_REGISTRY[call.tool_name].execution_policy
    result = execute_simulated_call(call, 0, run_budget_ms, policy)
    return {
        "label": label,
        "successes": int(result["success"]),
        "attempts": result["attempts"],
        "consumed_ms": result["end_ms"],
        "unused_ms": run_budget_ms - result["end_ms"],
        "run_budget_ms": run_budget_ms,
        "terminal": result["terminal"],
        "events": result["events"],
    }


retry_run = run_single_probe("bounded retry", retry_call, run_budget_ms=1500)
deadline_run = run_single_probe("deadline blocks retry", retry_call, run_budget_ms=900)
timeout_run = run_single_probe("terminal timeout", timeout_call, run_budget_ms=1600)
all_runs = comparison_runs + [retry_run, deadline_run, timeout_run]

print(f"{'policy':<24} {'success':>8} {'attempts':>10} {'consumed':>11} {'unused':>9}  terminal")
for run in all_runs:
    total_calls = 3 if run in comparison_runs else 1
    print(
        f"{run['label']:<24} {run['successes']}/{total_calls:>6} {run['attempts']:>10} "
        f"{run['consumed_ms']:>8} ms {run['unused_ms']:>6} ms  {run['terminal']}"
    )

assert comparison_runs[0]["successes"] == 1 and comparison_runs[0]["consumed_ms"] == 380
assert comparison_runs[1]["successes"] == 2 and comparison_runs[1]["consumed_ms"] == 1430
assert comparison_runs[2]["successes"] == 2 and comparison_runs[2]["consumed_ms"] == 880
assert retry_run["terminal"] == "completed" and retry_run["attempts"] == 2 and retry_run["consumed_ms"] == 850
assert deadline_run["terminal"] == "deadline_prevented_retry" and deadline_run["attempts"] == 1
assert timeout_run["terminal"] == "tool_timeout" and timeout_run["attempts"] == 2

colors = {
    "success": "#1d4ed8",
    "timeout": "#b91c1c",
    "transient_error": "#b45309",
    "backoff": "#64748b",
    "unused": "#15803d",
    "blocked": "#111827",
}
fig, axis = plt.subplots(figsize=(14, 7))
for row, run in enumerate(all_runs):
    axis.barh(row, run["unused_ms"], left=run["consumed_ms"], height=0.64, color=colors["unused"], alpha=0.24)
    for event in run["events"]:
        if event["kind"] == "blocked":
            axis.scatter(event["start_ms"], row, marker="x", s=90, color=colors["blocked"], linewidths=3, zorder=5)
            continue
        width = event["end_ms"] - event["start_ms"]
        color = colors[event["state"]]
        axis.barh(row, width, left=event["start_ms"], height=0.52, color=color)
        label = "backoff" if event["kind"] == "backoff" else f"{event['label']} A{event['attempt']}"
        if width >= 90:
            axis.text(event["start_ms"] + width / 2, row, label, ha="center", va="center", color="white", fontsize=7)
    axis.text(
        run["run_budget_ms"] + 25,
        row,
        f"{run['terminal']} | {run['attempts']} attempt(s)",
        va="center",
        fontsize=8,
    )

axis.set_yticks(range(len(all_runs)), [run["label"] for run in all_runs])
axis.invert_yaxis()
axis.set_xlim(0, max(run["run_budget_ms"] for run in all_runs) + 520)
axis.set_xlabel("Modeled run time (ms)")
axis.set_title("Tool attempts, backoff, consumed latency, and unused run budget")
axis.grid(axis="x", alpha=0.2)
axis.legend(
    handles=[
        Patch(color=colors["success"], label="successful attempt"),
        Patch(color=colors["transient_error"], label="retryable observation"),
        Patch(color=colors["timeout"], label="timed-out attempt"),
        Patch(color=colors["backoff"], label="backoff"),
        Patch(color=colors["unused"], alpha=0.24, label="unused run budget"),
    ],
    loc="lower right",
)
plt.tight_layout()
plt.show()

print("PASS: registered policies match the long global timeout's useful completions while consuming 550 ms less modeled latency.")
print("PASS: retries are classified, capped, charged for backoff, and refused when the next worst-case attempt cannot fit.")
print("PASS: repeated slow observations end in the explicit tool_timeout terminal state.")

**Code Walkthrough: policy ownership, admission, and evidence**

`ToolRegistration` owns an immutable `ToolExecutionPolicy`, so the model can choose a registered tool but cannot invent a more generous timeout. `execute_simulated_call` checks the remaining run budget against the next attempt's declared timeout plus any required backoff before either cost is incurred. The scripted observations then supply only modeled latency, outcome, category, and `retryable`; no wall clock or network behavior can make the result drift.

The Gantt resolves the prediction. A 150 ms global limit completes only the fast inventory call. A 700 ms global limit preserves the supplier quote but spends 1,430 ms because the stuck inventory call inherits the same allowance. Registered limits preserve the same two useful completions in 880 ms. The remaining lanes prove a successful bounded retry, a retry refused before it can overrun the deadline, and a final `tool_timeout` after two capped attempts.

**Your turn:** change `EXERCISE_RUN_BUDGET_MS` from `900` to `1100`. Predict whether the second supplier attempt is refused or allowed before running the next cell.

In [ ]:
# -- Your turn: change only the shared run budget ---------------------------
EXERCISE_RUN_BUDGET_MS = 900  # CHANGE THIS: try 1100
exercise_retry = run_single_probe(
    "exercise deadline",
    retry_call,
    run_budget_ms=EXERCISE_RUN_BUDGET_MS,
)

print(f"Run budget: {EXERCISE_RUN_BUDGET_MS} ms")
print(f"Attempts started: {exercise_retry['attempts']}")
print(f"Modeled latency consumed: {exercise_retry['consumed_ms']} ms")
print(f"Terminal state: {exercise_retry['terminal']}")
assert exercise_retry["consumed_ms"] <= EXERCISE_RUN_BUDGET_MS

if exercise_retry["terminal"] == "completed":
    print("PASS: the backoff and second worst-case attempt fit, so the retry was allowed.")
else:
    assert exercise_retry["terminal"] == "deadline_prevented_retry"
    print("PASS: the second attempt never started because its reservation did not fit.")

**Reflection:** Per-tool policy fixes the mismatch between fast inventory work and slower supplier work; it does not create extra run time. The shared deadline remains the outer authority, and `retryable=True` means only that a retry may be considered. Attempt caps, backoff, and worst-case admission still decide whether it can start.

**Checkpoint:** The deterministic comparison preserved 2/3 useful completions while reducing modeled latency from 1,430 ms to 880 ms. The controller also exposed `deadline_prevented_retry` and `tool_timeout` instead of hiding either condition behind another loop.

## 4 - Evaluate the Controller, Not Its Rationale

Completion means reaching a valid terminal state on solvable work. It does not mean the narrative sounded confident.

```mermaid
flowchart LR
    F["Fixture suite"] --> R["Run bounded controller"]
    R --> M["Measure success, steps, budget"]
    M --> G{ "Targets met?" }
    G -->|"Yes"| P["Promote controller"]
    G -->|"No"| D["Inspect failed trajectory"]
    style F fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style R fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style M fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style P fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Setup: import the deterministic OrderFlow runtime -----------------------
from pathlib import Path
import json
import sys


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "learning" / "agentic-ai" / "shared" / "__init__.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from the ai-portfolio repository or a descendant directory.")


REPO_ROOT = find_repo_root()
TRACK_DIR = REPO_ROOT / "learning" / "agentic-ai"
if str(TRACK_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_DIR))
from shared import load_purchase_requests

solvable = [request for request in load_purchase_requests() if request["solvable"]]
results = []
for request in solvable:
    try:
        run = run_bounded(request)
        success = run.terminal == "completed"
        steps = len(run.trace)
    except (KeyError, ValueError):
        success = False
        steps = 0
    results.append({"request_id": request["request_id"], "success": success, "steps": steps})

success_rate = sum(row["success"] for row in results) / len(results)
max_steps = max(row["steps"] for row in results)
print(f"Solvable completion: {sum(row['success'] for row in results)}/{len(results)} ({success_rate:.0%})")
print(f"Maximum observed steps: {max_steps}")
assert success_rate >= 0.90 and max_steps <= 8
print("PASS: bounded control preserves at least 90% solvable completion and terminates within eight steps.")

**Your turn:** set `EXERCISE_MAX_STEPS` to `2`. The same task should stop at `step_budget_exhausted`, proving the budget is enforced rather than documented.


In [ ]:
# -- Your turn: change one budget -----------------------------------------
EXERCISE_MAX_STEPS = 8  # CHANGE THIS: try 2
exercise = run_bounded(shortage_request, Budget(max_steps=EXERCISE_MAX_STEPS))
print(f"Terminal state with max_steps={EXERCISE_MAX_STEPS}: {exercise.terminal}")


## Roadmap Checkpoint

| Constraint | Before | After |
|---|---:|---:|
| Repeated inventory calls | 12 before external stop | Cycle detectable from the second repeat |
| Terminal bound | None | At most 8 steps |
| Solvable fixture completion | Not measured | At least 90%, computed above |
| Tool timeout fit | One global value either kills slow work or indulges stuck work | Registered 150 ms inventory and 700 ms supplier limits preserve 2/3 useful completions |
| Modeled latency for those 2/3 completions | 1,430 ms with the long global timeout | 880 ms with per-tool policy; 550 ms less |
| Retry and deadline behavior | Retryability described but not enforced | Two-attempt cap, 100 ms backoff, pre-attempt deadline admission, `deadline_prevented_retry`, and `tool_timeout` |

```mermaid
flowchart LR
    A["Bounded controller"] --> B["Next failure: context mixes POs"]
    B --> C["Notebook 02: state and memory"]
    style A fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Coverage Ledger

| Tier | Covered here |
|---|---|
| Built and measured | Explicit plan, step/token/cost budgets, cycle detection, typed per-tool timeout/retry policy, modeled deadline accounting, bounded backoff, and terminal states |
| Explained and illustrated | Progress checks and the boundary between Chapter 00 error classification and this chapter's simple retryability decision |
| Named with a reason | Hidden chain-of-thought, excluded because control must be inspectable |

### Key Takeaways

- Reasoning quality cannot substitute for a terminal-state design.
- Timeout and retry policy belongs to tool registration, not model improvisation.
- A retryable observation is eligible for another attempt only when the cap and remaining run budget permit it.
- Reserve the next attempt's declared worst case before starting backoff or work.
- A structural fingerprint catches exact loops before the budget disappears.
- Evaluate completion, attempts, consumed latency, and terminal state, not confidence of prose.